# Data preprocessing

Combine saved hourly datasets into a numeric feature table and a capacity-factor target.

In [4]:
START_DATE = "2022-01-01"
END_DATE = "2026-10-18"
INPUT_DIR = f"data/raw/{START_DATE}_{END_DATE}"
OUTPUT_FILE = f"data/processed/dataset_{START_DATE}_{END_DATE}.parquet"

In [5]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
DATA_PATH = ROOT / INPUT_DIR
OUTPUT_PATH = ROOT / OUTPUT_FILE

## Load saved data

In [6]:
smard = pd.read_parquet(DATA_PATH / "smard.parquet")
weather = pd.read_parquet(DATA_PATH / "weather.parquet")
capacity = pd.read_parquet(DATA_PATH / "mastr.parquet")
calendar = pd.read_parquet(DATA_PATH / "calendar.parquet")
solar = pd.read_parquet(DATA_PATH / "solar.parquet")

## Validate the common hourly interval

Stop on incomplete inputs instead of silently dropping rows or filling values.

In [7]:
hours = pd.date_range(
    START_DATE,
    pd.Timestamp(END_DATE) + pd.Timedelta(days=1),
    freq="h",
    inclusive="left",
    tz="UTC",
).as_unit("ns")
datasets = {
    "SMARD": smard,
    "Weather": weather,
    "Capacity": capacity,
    "Calendar": calendar,
    "Solar": solar,
}
for name, frame in datasets.items():
    frame["timestamp_utc"] = pd.to_datetime(
        frame["timestamp_utc"], utc=True
    ).dt.as_unit("ns")
    keys = ["timestamp_utc", "location"] if name == "Weather" else ["timestamp_utc"]
    assert not frame[keys].isna().any().any(), name
    assert not frame.duplicated(keys).any(), name
    timestamps = pd.DatetimeIndex(frame.timestamp_utc.unique()).sort_values()
    assert timestamps.equals(hours), name
    assert np.isfinite(frame.select_dtypes(include="number").to_numpy()).all(), name

assert weather.location.nunique() == 10
assert weather.groupby("location").size().eq(len(hours)).all()
assert smard.target_solar_generation_mw.ge(0).all()
assert capacity.installed_pv_capacity_mw.gt(0).all()

## Aggregate weather

Use equal-weight means across the ten locations and maxima for cloud-cover variables. These weights are a baseline, not installed-capacity weights.

In [8]:
weather_columns = (
    weather.select_dtypes(include="number")
    .columns.difference(["latitude", "longitude"])
    .tolist()
)
cloud_columns = [
    column for column in weather_columns if column.startswith("cloud_cover")
]
grouped = weather.groupby("timestamp_utc")
weather_mean = (
    grouped[weather_columns].mean().add_prefix("weather_").add_suffix("_mean")
)
weather_max = grouped[cloud_columns].max().add_prefix("weather_").add_suffix("_max")
weather_hourly = weather_mean.join(weather_max).reset_index()
weather_hourly.head()

,timestamp_utc,weather_cloud_cover_mean,weather_cloud_cover_high_mean,weather_cloud_cover_low_mean,weather_cloud_cover_mid_mean,weather_diffuse_radiation_mean,weather_direct_radiation_mean,weather_precipitation_mean,weather_shortwave_radiation_mean,weather_temperature_2m_mean,weather_wind_speed_10m_mean,weather_cloud_cover_max,weather_cloud_cover_high_max,weather_cloud_cover_low_max,weather_cloud_cover_mid_max
0,2022-01-01 00:00:00+00:00,92.3,65.9,81.7,32.6,0.0,0.0,0.02,0.0,10.85,17.15,100,98,100,100
1,2022-01-01 01:00:00+00:00,95.1,60.0,84.4,28.9,0.0,0.0,0.06,0.0,10.80,17.09,100,100,100,100
2,2022-01-01 02:00:00+00:00,95.8,73.6,78.0,21.0,0.0,0.0,0.07,0.0,10.70,16.42,100,99,100,97
3,2022-01-01 03:00:00+00:00,92.3,62.4,85.9,31.1,0.0,0.0,0.05,0.0,10.52,15.74,100,99,100,100
4,2022-01-01 04:00:00+00:00,93.1,68.3,74.2,18.0,0.0,0.0,0.03,0.0,10.42,14.98,100,100,100,94


## Join features and generation

Keep numeric calendar and solar features. Preserve timestamps as metadata, outside the model inputs.

In [9]:
calendar_columns = calendar.select_dtypes(include="number").columns.tolist()
dataset = smard.copy()
for frame in [
    capacity,
    calendar[["timestamp_utc", *calendar_columns]],
    solar,
    weather_hourly,
]:
    dataset = dataset.merge(
        frame, on="timestamp_utc", how="left", validate="one_to_one"
    )
dataset = dataset.sort_values("timestamp_utc").reset_index(drop=True)
dataset.head()

,target_solar_generation_mwh,timestamp_utc,target_solar_generation_mw,installed_pv_capacity_mw,hour_local,day_of_year,month,weekday,is_weekend,is_holiday_de,...,weather_diffuse_radiation_mean,weather_direct_radiation_mean,weather_precipitation_mean,weather_shortwave_radiation_mean,weather_temperature_2m_mean,weather_wind_speed_10m_mean,weather_cloud_cover_max,weather_cloud_cover_high_max,weather_cloud_cover_low_max,weather_cloud_cover_mid_max
0,2.0,2022-01-01 00:00:00+00:00,2.0,54499.407232,1,1,1,5,1,1,...,0.0,0.0,0.02,0.0,10.85,17.15,100,98,100,100
1,2.0,2022-01-01 01:00:00+00:00,2.0,54499.407232,2,1,1,5,1,1,...,0.0,0.0,0.06,0.0,10.80,17.09,100,100,100,100
2,2.0,2022-01-01 02:00:00+00:00,2.0,54499.407232,3,1,1,5,1,1,...,0.0,0.0,0.07,0.0,10.70,16.42,100,99,100,97
3,2.0,2022-01-01 03:00:00+00:00,2.0,54499.407232,4,1,1,5,1,1,...,0.0,0.0,0.05,0.0,10.52,15.74,100,99,100,100
4,2.0,2022-01-01 04:00:00+00:00,2.0,54499.407232,5,1,1,5,1,1,...,0.0,0.0,0.03,0.0,10.42,14.98,100,100,100,94


## Define the target and model inputs

Capacity factor is mean generation divided by installed capacity. Multiply predictions by capacity to recover MW. Keep both original generation columns for evaluation, excluding them from model inputs.

Inputs are joined on unchanged `timestamp_utc` values. Targets are not clipped; no night-time overrides or radiation/clear-sky ratio are applied. Scaling, feature selection, and imputation must be fitted on training data only.

In [10]:
TARGET = "target_solar_capacity_factor"
dataset[TARGET] = dataset.target_solar_generation_mw / dataset.installed_pv_capacity_mw
FEATURES = [
    column
    for column in dataset.select_dtypes(include="number").columns
    if not column.startswith("target_")
]
assert dataset[TARGET].between(0, 1).all()
assert np.isfinite(dataset[[*FEATURES, TARGET]].to_numpy()).all()
assert len(dataset) == len(hours)
assert pd.DatetimeIndex(dataset.timestamp_utc).equals(hours)
print(f"Rows: {len(dataset):,}; features: {len(FEATURES)}; target: {TARGET}")
pd.Series(FEATURES, name="feature")

Rows: 42,048; features: 33; target: target_solar_capacity_factor


0             installed_pv_capacity_mw
1                           hour_local
2                          day_of_year
3                                month
4                              weekday
5                           is_weekend
6                        is_holiday_de
7                             hour_sin
8                             hour_cos
9                      day_of_year_sin
10                     day_of_year_cos
11                solar_elevation_mean
12          solar_apparent_zenith_mean
13                  clear_sky_ghi_mean
14                  clear_sky_dni_mean
15                  clear_sky_dhi_mean
16                 solar_elevation_max
17                   clear_sky_ghi_max
18                is_daylight_anywhere
19            weather_cloud_cover_mean
20       weather_cloud_cover_high_mean
21        weather_cloud_cover_low_mean
22        weather_cloud_cover_mid_mean
23      weather_diffuse_radiation_mean
24       weather_direct_radiation_mean
25          weather_preci

## Save the training table

This table is not a train/test split. Use chronological splits in the training notebook. Historical forecasts and reconstructed capacity retain the limitations identified during analysis.

In [11]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
temporary = OUTPUT_PATH.with_suffix(".tmp")
dataset.to_parquet(temporary, index=False)
temporary.replace(OUTPUT_PATH)

PosixPath('/Users/denys/Projects/Academic/masters-thesis/data/processed/dataset_2022-01-01_2026-10-18.parquet')

In [12]:
saved = pd.read_parquet(OUTPUT_PATH)
pd.testing.assert_frame_equal(saved, dataset)
saved.head()

,target_solar_generation_mwh,timestamp_utc,target_solar_generation_mw,installed_pv_capacity_mw,hour_local,day_of_year,month,weekday,is_weekend,is_holiday_de,...,weather_direct_radiation_mean,weather_precipitation_mean,weather_shortwave_radiation_mean,weather_temperature_2m_mean,weather_wind_speed_10m_mean,weather_cloud_cover_max,weather_cloud_cover_high_max,weather_cloud_cover_low_max,weather_cloud_cover_mid_max,target_solar_capacity_factor
0,2.0,2022-01-01 00:00:00+00:00,2.0,54499.407232,1,1,1,5,1,1,...,0.0,0.02,0.0,10.85,17.15,100,98,100,100,0.000037
1,2.0,2022-01-01 01:00:00+00:00,2.0,54499.407232,2,1,1,5,1,1,...,0.0,0.06,0.0,10.80,17.09,100,100,100,100,0.000037
2,2.0,2022-01-01 02:00:00+00:00,2.0,54499.407232,3,1,1,5,1,1,...,0.0,0.07,0.0,10.70,16.42,100,99,100,97,0.000037
3,2.0,2022-01-01 03:00:00+00:00,2.0,54499.407232,4,1,1,5,1,1,...,0.0,0.05,0.0,10.52,15.74,100,99,100,100,0.000037
4,2.0,2022-01-01 04:00:00+00:00,2.0,54499.407232,5,1,1,5,1,1,...,0.0,0.03,0.0,10.42,14.98,100,100,100,94,0.000037
